In [ ]:
import sys
sys.path.append("/kaggle/input/datasets/yazidsultan/xail-core")
sys.path.append("/kaggle/input/datasets/yazidsultan/vinbig")

import gc
import json
from pathlib import Path

import pandas as pd
import torch 
import torch.nn.functional as F
from torch import optim
from torch.optim.lr_scheduler import StepLR

from vinbig_prep import open_dataset
from core.config import (
    DataConfig, ExperimentConfig, ExplanationLossConfig, ModelConfig,
    RunConfig, SplitConfig, TrainConfig,
)
from core.data import DiseaseTaskBuilder
from core.datasets import BinaryDiseaseDataset,ProcessedDataset, build_dataloaders
from core.engine import Trainer
from core.metrics import MetricBundle, ExplanationMetricBundle
from core.losses import CombinedLoss, ExplanationLoss
from core.model import build_model
from core.visualize import plot_history
from core.seeding import set_seed

In [ ]:
# experiment configuration
disease = "Pulmonary fibrosis"
alpha = 1.0         # the weight of the explaination loss
seed = 42

for alpha in (0.25, 0.5, 0.75, 1):
    process_path = Path('/kaggle/input/notebooks/yazidsultan/vingbig-preprocessing/processed')
    data_config = DataConfig(processed_dir=process_path)
    run_config = RunConfig(runs_dir=Path(f'./runs_alpha_{alpha}/'))
    train_config = TrainConfig(
        epochs = 60, 
        lr = 2e-4,
        weight_decay = 1e-4,
        train_batch_size = 12,
        eval_batch_size = 12,
        num_workers=1,
        seed = seed,
        device = "auto"  # auto: use cuda if available, otherwise use cpu
    )
    split_config = SplitConfig(
        val_fraction = 0.1,
        test_fraction = 0.1,
        seed = seed
    )
    
    # the default model configurations used in the original paper 
    model_config = ModelConfig(
        dropout = 0.3,
        num_classes = 2,
        pretrained = True,
        feature_node = "features.denseblock4",
        logits_node = "classifier"
    )
    
    exp_config = ExplanationLossConfig(
        enabled = True,       # use explanation loss in the training
        alpha = alpha,        # the weight of the explanation loss
        quantile = 1,       # the top largest #qunatile of the gradients to consider in the loss
        temperature = 0.05,    # temperature for the soft masking.
        score_mode = "sqr",   # alg: z1 - z0, abs: |z1 - z0|, sqr: (z1-z0)^2 z1, z0 is the logits of positive class and negative class, respectivly.
        use_probs = False,    # use probabilities instead of logits (applies sigmoid to logits)
        only_positive_samples = True # apply explination to positive class only (recommendation: always keep True)
    )
    
    
    config = ExperimentConfig(
        data = data_config,
        split = split_config,
        model = model_config,
        explanation_loss = exp_config,
        train = train_config,
        run = run_config,
        disease=disease
    )
    
    
    set_seed(config.train.seed)
    # Read the preprocessed data (disease list/order, image size, dtypes)
    dataset = open_dataset(config.data.processed_dir)
    
    # build disease specific splits train_ids, val_ids, test_ids for the specified disease conting positive and negative ids
    task_builder = DiseaseTaskBuilder(dataset, config.split)
    task = task_builder.build(config.disease)
    print(
        f"{config.disease}: train={len(task['train_ids'])}, "
        f"val={len(task['val_ids'])}, test={len(task['test_ids'])}"
    )
    
    disease_idx = dataset.metadata.disease_to_idx[config.disease]
    train_dl, val_dl, test_dl = build_dataloaders(task, dataset, disease_idx, config.train)
    device = config.train.resolve_device()
    
    model = build_model(config.model).to(device)
    optimizer = optim.Adam(model.parameters(), lr=config.train.lr, weight_decay=config.train.weight_decay)
    loss_fn = CombinedLoss(config.explanation_loss)
    scheduler = StepLR(optimizer, step_size = 10, gamma = 0.5)
    trainer = Trainer(model, loss_fn, optimizer, scheduler, config.train, config.model, config.run, config.disease)
    history = trainer.fit(train_dl, val_dl)
    plot_history(history, trainer.output_dir, show=True)
    
    # free gpu memeory
    del trainer
    del optimizer
    gc.collect()
    torch.cuda.empty_cache()
    
    cls_metrics = MetricBundle(num_classes=2,device=device)
        
    model.eval()
    for images, labels, masks in test_dl:
            images = images.to(device)
            labels = labels.to(device)
            masks = masks.to(device)
        
            logits, feature_map = model(images)
            loss = F.cross_entropy(logits, labels)
        
            cls_metrics.update(loss.detach(), logits.detach(), labels)
            
    results = {f"test_{k}": v for k, v in cls_metrics.compute().items()}
    json_path = run_config.disease_dir(disease) / 'test_results.json'
    with open(json_path, "w") as f:
        json.dump(results, f, indent=4)
            
    print(f"\nTest results Alpha({alpha}):")
    for k, v in results.items():
        print(f"  {k}: {v:.4f}")

In [ ]:
def evaluate(model):
    cls_metrics = MetricBundle(num_classes=2,device=device)
    explanation_metrics = ExplanationMetricBundle(
        device,
        quantile=exp_config.quantile,
        temperature=exp_config.temperature,
        score_mode=exp_config.score_mode,
        use_probs=exp_config.use_probs,
    )
    
    model.eval()
    for images, labels, masks in test_dl:
            images = images.to(device)
            labels = labels.to(device)
            masks = masks.to(device)
    
            logits, feature_map = model(images)
            loss = F.cross_entropy(logits, labels)
    
            cls_metrics.update(loss.detach(), logits.detach(), labels)
            explanation_metrics.update(logits, feature_map, masks)
        
    results = {f"test_{k}": v for k, v in cls_metrics.compute().items()}
    results.update(explanation_metrics.compute())
    json_path = run_config.disease_dir(disease) / 'test_results.json'
    with open(json_path, "w") as f:
        json.dump(results, f, indent=4)
        
    print("\nTest results:")
    for k, v in results.items():
        print(f"  {k}: {v:.4f}")

In [ ]:
def compute_heatmap(explainer: ExplanationLoss, logits: torch.Tensor, feature_map: torch.Tensor) -> torch.Tensor:
    """Full-resolution, unthresholded normalized Grad-CAM heatmap (H_hat).
    Split out from `ExplanationLoss.get_heatmaps` (which applies the
    top-k mask immediately) since here we want to visualize both the raw
    heatmap AND the top-k mask separately.

    create_graph=False: this is a pure visualization pass, nothing here
    gets backpropagated, so there's no need for the training-time
    double-backward graph (see the create_graph discussion in
    xai_train/losses.py's gradcam_gradients).
    """
    scores = explainer.classification_score(logits)
    gradients = explainer.gradcam_gradients(scores, feature_map)
    weights = explainer.gradcam_weights(gradients)
    heatmap = explainer.gradcam_heatmap(weights, feature_map)
    return explainer.minmax_normalize(heatmap).detach()


def upsample(x: torch.Tensor, size: int, mode: str) -> torch.Tensor:
    """(B, h, w) -> (B, size, size). `mode="nearest"` for binary masks
    (preserve exact 0/1), `mode="bilinear"` for the continuous heatmap
    (smooth interpolation is fine/desirable for visualization)."""
    kwargs = {"mode": mode}
    if mode not in ("nearest", "nearest-exact"):
        kwargs["align_corners"] = False
    return F.interpolate(x.unsqueeze(1).float(), size=(size, size), **kwargs).squeeze(1)


def collect_annotated_samples(test_dl, n_samples: int, device: str):
    """Walks test_dl until n_samples annotated (mask-positive) examples
    are collected -- test_dl is 1:1 positive:negative, so this skips the
    negatives (which have an all-zero mask and nothing to visualize
    against)."""
    images_list, masks_list = [], []
    collected = 0
    for images, _labels, masks in test_dl:
        has_annotation = masks.flatten(1).any(dim=1)
        if has_annotation.any():
            images_list.append(images[has_annotation])
            masks_list.append(masks[has_annotation])
            collected += int(has_annotation.sum())
        if collected >= n_samples:
            break

    if not images_list:
        raise RuntimeError(
            "No annotated samples found in the test split for this disease -- "
            "check --disease and --seed match what you trained with."
        )

    images = torch.cat(images_list)[:n_samples]
    masks = torch.cat(masks_list)[:n_samples]
    return images.to(device), masks.to(device)


def plot_grid(images: torch.Tensor, masks: torch.Tensor, heatmap_up: torch.Tensor, topk_up: torch.Tensor, save_path: Path):
    n = images.shape[0]
    fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n), squeeze=False)

    col_titles = ["Image + GT box", "Grad-CAM heatmap", "Top-k saliency mask"]
    for i in range(n):
        img = images[i, 0].cpu().numpy()  # channel 0; all 3 channels are identical (grayscale expanded)
        mask = masks[i].cpu().numpy()
        heat = heatmap_up[i].cpu().numpy()
        topk = topk_up[i].cpu().numpy()

        axes[i, 0].imshow(img, cmap="gray")
        axes[i, 1].imshow(img, cmap="gray")
        axes[i, 1].imshow(heat, cmap="jet", alpha=0.45, vmin=0, vmax=1)
        axes[i, 2].imshow(img, cmap="gray")
        axes[i, 2].imshow(topk, cmap="Reds", alpha=0.45, vmin=0, vmax=1)

        for ax in axes[i]:
            if mask.max() > 0:
                ax.contour(mask, colors="lime", linewidths=1.5, levels=[0.5])
            ax.axis("off")
        if i == 0:
            for ax, title in zip(axes[i], col_titles):
                ax.set_title(title)

    fig.tight_layout()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"Saved: {save_path}")
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
n_samples=8
save_path = Path('./grad_cam.png')

model.eval()
images, masks = collect_annotated_samples(test_dl, n_samples, device)
print(f"Visualizing {images.shape[0]} annotated test samples for '{disease}'")

explainer = ExplanationLoss(quantile=exp_config.quantile, score_mode=exp_config.score_mode, temperature=exp_config.temperature, use_probs=exp_config.use_probs)
logits, feature_map = model(images)  # grad-enabled forward -- required for Grad-CAM, do NOT wrap in no_grad

heatmap = compute_heatmap(explainer, logits, feature_map)  # (B, h, w) continuous, e.g. 7x7 for DenseNet121@224
#topk = explainer.soft_mask(heatmap)  # (B, h, w) binary
topk = heatmap
img_size = images.shape[-1]
heatmap_up = upsample(heatmap, img_size, mode="bilinear")
topk_up = upsample(topk, img_size, mode="nearest")

plot_grid(images, masks, heatmap_up, topk_up, save_path)